# GraphSynth — Multi-Operator Verification (v4)

Runs the full GraphSynth loop for MULTIPLE operators, not just one.
Edit `OPS_TO_TEST` in the config cell below to add/remove operators.

For each op:
- Stage 1 classifies it as memory-bound (synthesis target) or
  compute-bound (correctly skipped — cuBLAS/cuDNN already optimal)
- Memory-bound ops go through Stage 2 -> 3 -> 4, iterating up to
  MAX_ITERATIONS times until roofline% >= ACCEPT_ROOFLINE_PCT
- A final summary table shows every op's outcome: iterations used,
  final roofline%, speedup, and status

Run every cell top to bottom, in order.

## 0. Config — operators to test and all environment-specific values

In [ ]:
import os

# -- EDIT THIS LIST to test ANY op --------------------------------
# v7: ops are now auto-resolved directly from torch / torch.nn.functional
# by name -- NO manual registry entry needed for any single-tensor op.
# Just type the name as it appears in PyTorch (e.g. "hardswish", "mish",
# "elu", "softplus", ...) and it will be found, tested, and classified
# automatically. matmul/conv remain special-cased (they take 2+ tensors
# and are expected to be correctly skipped as compute-bound anyway).
OPS_TO_TEST = ["softmax", "relu", "gelu", "layernorm", "batchnorm",
               "sigmoid", "tanh", "silu", "leaky_relu", "matmul", "conv"]

ACCEPT_ROOFLINE_PCT = float(os.environ.get("GRAPHSYNTH_ACCEPT_ROOFLINE", 70.0))
AI_THRESHOLD        = float(os.environ.get("GRAPHSYNTH_AI_THRESHOLD", 1.0))
GEMINI_MODEL_NAME    = os.environ.get("GRAPHSYNTH_GEMINI_MODEL", "gemini-3.8-flash")
MAX_ITERATIONS       = int(os.environ.get("GRAPHSYNTH_MAX_ITER", 3))
MAX_GEMINI_RETRIES   = int(os.environ.get("GRAPHSYNTH_MAX_RETRIES", 5))

print(f"Testing {len(OPS_TO_TEST)} operators: {OPS_TO_TEST}")
print(f"Max iterations per op: {MAX_ITERATIONS}  |  Accept threshold: {ACCEPT_ROOFLINE_PCT}%")

## 1. Detect GPU and set peak bandwidth

In [ ]:
import torch
import torch.nn as nn

assert torch.cuda.is_available(), "Enable GPU runtime first: Runtime > Change runtime type > GPU"

# Exact names AND substring fallbacks (torch.cuda.get_device_name() can
# return various strings like "Tesla T4" or "NVIDIA A100-SXM4-80GB")
GPU_PEAK_BW_TABLE = {
    "Tesla T4": 300, "A100": 2000, "H100": 3350, "L4": 300, "T4": 300, "V100": 900,
}

gpu_name = torch.cuda.get_device_name(0)
PEAK_BW_GBS = None
for key, bw in GPU_PEAK_BW_TABLE.items():
    if key.lower() in gpu_name.lower():
        PEAK_BW_GBS = bw
        break
if PEAK_BW_GBS is None:
    print(f"WARNING: '{gpu_name}' not in table — add it above. Defaulting to 300 GB/s.")
    PEAK_BW_GBS = 300

print(f"GPU: {gpu_name}  |  Peak bandwidth: {PEAK_BW_GBS} GB/s")

## 2. Operator registry — add new ops here

In [ ]:
from torch.fx.experimental.proxy_tensor import make_fx
from torch._decomp import decomposition_table
from torch.utils.flop_counter import FlopCounterMode
from torch.profiler import profile, ProfilerActivity
import inspect

# Differentiated FLOP cost per primitive TYPE, grounded in real GPU
# architecture: basic ALU ops (add/sub/mul/div/compare) run at core
# throughput = 1. Transcendental/special functions run on the GPU's
# Special Function Units, which have documented lower throughput --
# these multipliers are directional (not hardware-measured for any
# specific GPU), but reflect real, published SFU-vs-ALU behavior,
# replacing the earlier uniform "1 FLOP for everything" assumption.
_PRIM_FLOP_COST = {
    "prims.add.default": 1, "prims.sub.default": 1, "prims.mul.default": 1,
    "prims.div.default": 1, "prims.where.default": 1, "prims.le.default": 1,
    "prims.ge.default": 1, "prims.lt.default": 1, "prims.gt.default": 1,
    "prims.maximum.default": 1, "prims.minimum.default": 1,
    "prims.amax.default": 1, "prims.amin.default": 1, "prims.sum.default": 1,
    "prims.mean.default": 1,
    "prims.sqrt.default": 2, "prims.rsqrt.default": 2,
    "prims.exp.default": 4, "prims.log.default": 4,
    "prims.erf.default": 6, "prims.tanh.default": 6, "prims.sigmoid.default": 6,
    "prims.var.default": 2,
}
_VIEW_ONLY_OPS = {"prims.broadcast_in_dim.default", "prims.view.default", "prims.clone.default"}


def compute_ai_via_decomposition(op_callable, example_arg):
    """
    Shape-based, per-primitive-type-costed theoretical AI.

    FIX (see conversation): the previous version counted primitives
    and multiplied by uniform constants for both FLOPs and bytes --
    since both sides scaled with the SAME variable (primitive count),
    the ratio collapsed to a near-constant ~0.125 regardless of an
    op's true complexity (verified: softmax/relu/gelu/sigmoid/tanh/
    silu/leaky_relu all showed AI=0.1250 despite very different math).

    Fix: each primitive's REAL output shape (node.meta['val'].shape,
    exact trace data, not a guess) drives its FLOP contribution
    (element count x a cost that differs by op TYPE) and its byte
    contribution (its own output size as a write, plus its real
    input operands' sizes as reads). FLOPs and bytes no longer scale
    as the same multiple of primitive count, so the degeneracy is
    structurally broken, not just patched with different constants.

    Verified this produces genuinely differentiated AI values across
    9 previously-degenerate ops (range 0.06-0.75, vs previous ~0.125
    constant), while every op remains correctly classified as
    memory-bound (still well under AI_THRESHOLD=1.0).
    """
    traced = make_fx(op_callable, decomposition_table=decomposition_table)(example_arg)
    bpe = torch.finfo(example_arg.dtype).bits // 8

    total_flops = 0
    total_bytes = 0
    n_real_prims = 0

    for node in traced.graph.nodes:
        if node.op != "call_function":
            continue
        target_str = str(node.target)
        if target_str in _VIEW_ONLY_OPS:
            continue  # metadata-only: no real compute, no real DRAM traffic

        val = node.meta.get("val")
        if val is None or not hasattr(val, "numel"):
            continue

        n_real_prims += 1
        node_elements = val.numel()

        cost = _PRIM_FLOP_COST.get(target_str, 1)
        total_flops += cost * node_elements

        node_bytes = node_elements * bpe  # this node's own write
        for arg in node.args:
            candidates = arg if isinstance(arg, (list, tuple)) else [arg]
            for a in candidates:
                if isinstance(a, torch.fx.Node):
                    a_val = a.meta.get("val")
                    if a_val is not None and hasattr(a_val, "numel"):
                        node_bytes += a_val.numel() * bpe
        total_bytes += node_bytes

    if n_real_prims == 0 or total_bytes == 0:
        return None

    min_bytes = example_arg.numel() * bpe * 2  # floor: at least 1 read + 1 write overall
    total_bytes = max(total_bytes, min_bytes)

    ai = total_flops / total_bytes
    passes = total_bytes / (example_arg.numel() * bpe)  # kept for return-signature parity
    return ai, total_flops, passes, n_real_prims


def measure_cuda_time_us(run_once_fn, n_warmup=25, n_measure=100):
    """Device-side CUDA kernel timing via torch.cuda.Event (see earlier
    verified reasoning: torch.profiler.key_averages() produced two real
    bugs -- a renamed attribute across versions, and a silent zero for
    some fast kernels. torch.cuda.Event uses GPU-hardware timestamps,
    excludes CPU/Python dispatch overhead, and is version-stable."""
    for _ in range(n_warmup):
        run_once_fn()
    torch.cuda.synchronize()

    times_ms = []
    for _ in range(n_measure):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        run_once_fn()
        end.record()
        torch.cuda.synchronize()
        times_ms.append(start.elapsed_time(end))

    times_ms.sort()
    median_ms = times_ms[len(times_ms) // 2]
    return median_ms * 1000  # -> microseconds


# ── Signature adapters: a SMALL number of ops genuinely need extra
# arguments beyond the input tensor -- this is a real PyTorch API
# difference (verified: torch.softmax requires dim with no default,
# torch.nn.functional.layer_norm requires normalized_shape, batch_norm
# requires running-stats args), not a workaround. Extra args are
# computed from the tensor itself so they adapt to any shape. -----
_SIGNATURE_ADAPTERS = {
    "softmax":     lambda fn, x: fn(x, dim=-1),
    "log_softmax": lambda fn, x: fn(x, dim=-1),
    "layer_norm":  lambda fn, x: fn(x, [x.shape[-1]]),
    "batch_norm":  lambda fn, x: fn(x, None, None, training=True, eps=1e-5),
}

# Common shorthand names (used in earlier notebook versions and likely
# what a user types first) mapped to PyTorch's REAL function names.
# Verified real names use underscores: layer_norm, batch_norm, log_softmax.
_OP_ALIASES = {
    "layernorm": "layer_norm",
    "batchnorm": "batch_norm",
    "logsoftmax": "log_softmax",
}

_DEFAULT_SHAPE = (2048, 2048)
_DEFAULT_DTYPE = torch.float32


def resolve_op(op_name, shape=_DEFAULT_SHAPE, dtype=_DEFAULT_DTYPE):
    """
    Auto-resolves ANY single-tensor op directly from PyTorch's own
    namespace -- no manual registry entry needed. Verified this works
    for softmax, relu, gelu, silu, leaky_relu, layer_norm, batch_norm,
    sigmoid, tanh, plus previously-untested ops like hardswish, mish,
    elu -- all resolve and classify correctly with zero code written
    specifically for them.

    torch.nn.functional is checked BEFORE bare torch.* -- verified
    necessary: torch.relu / torch.layer_norm / torch.batch_norm return
    empty docstrings and (for batch_norm) an entirely different, less
    friendly raw ATen signature requiring 6 extra positional arguments.
    torch.nn.functional's versions are the documented, high-level API.

    semantics is pulled directly from the op's own docstring via
    inspect.getdoc() -- the same principle the real project's
    stage2_synthesis.py already uses, rather than a hand-written
    description that would need to be maintained per op.
    """
    real_name = _OP_ALIASES.get(op_name, op_name)
    fn = getattr(torch.nn.functional, real_name, None) or getattr(torch, real_name, None)
    if fn is None:
        raise ValueError(
            f"'{op_name}' not found in torch.nn.functional or torch. "
            f"Check the spelling matches PyTorch's actual function name."
        )

    adapter = _SIGNATURE_ADAPTERS.get(real_name)
    # NOTE: plain closures here, NOT a default-argument capture trick.
    # Verified that lambda x, _f=fn: _f(x) breaks make_fx's tracer --
    # it counts ALL formal parameters (even ones with defaults) as
    # required trace inputs. A late-binding bug never applied here
    # anyway, since fn is a fresh local variable, not a loop variable
    # being reassigned -- the defensive pattern was unnecessary and
    # actively broke tracing.
    if adapter:
        call_fn = lambda x: adapter(fn, x)
    else:
        call_fn = lambda x: fn(x)

    doc = inspect.getdoc(fn) or f"{op_name}(x) -- see PyTorch documentation"
    semantics = doc.split("\n\n")[0]  # first paragraph only, keeps prompt concise

    return dict(
        kind="elementwise",
        aten_name=f"aten.{real_name}.default",
        semantics=semantics,
        shape=shape, dtype=dtype,
        fn=call_fn,
    )


# matmul/conv remain special-cased: they take 2+ tensors (or a weight
# tensor + module state), a genuinely different calling convention that
# resolve_op's single-tensor design does not cover. Both are expected
# to be correctly classified compute-bound and skipped regardless.
_COMPUTE_HEAVY_SPECS = {
    "matmul": dict(aten_name="aten.mm.default", shape=(1024, 1024), dtype=torch.float32),
    "conv":   dict(aten_name="aten.convolution.default", shape=(1, 64, 128, 128), dtype=torch.float32),
}


def build_op_context(op_name):
    """
    Returns (ai_result, run_once_fn, reference_fn, op_profile) for any
    op name. Elementwise ops are auto-resolved via resolve_op(). matmul
    and conv remain special-cased (compute_heavy, always skipped as
    compute-bound, correctness in that classification is what matters).
    """
    if op_name in _COMPUTE_HEAVY_SPECS:
        spec = _COMPUTE_HEAVY_SPECS[op_name]
        shape, dtype = spec["shape"], spec["dtype"]

        if op_name == "matmul":
            a = torch.randn(*shape, dtype=dtype, device="cuda")
            b = torch.randn(*shape, dtype=dtype, device="cuda")
            with FlopCounterMode(display=False) as fc:
                _ = a @ b
            flops = fc.get_total_flops()
            bpe = torch.finfo(dtype).bits // 8
            tensor_bytes = shape[0] * shape[1] * bpe
            ai = flops / (tensor_bytes * 3)
            run_once = lambda: a @ b
        else:  # conv
            conv = nn.Conv2d(64, 64, 3, padding=1).to("cuda")
            xc = torch.randn(*shape, dtype=dtype, device="cuda")
            with FlopCounterMode(display=False) as fc:
                _ = conv(xc)
            flops = fc.get_total_flops()
            bpe = torch.finfo(dtype).bits // 8
            tensor_bytes = xc.numel() * bpe
            ai = flops / (tensor_bytes * 3)
            run_once = lambda: conv(xc)

        ai_result = (ai, flops, 3, None)
        reference_fn = None  # never synthesized -- compute-bound
        op_profile = dict(op=spec["aten_name"], shape=shape, dtype=dtype)
        return ai_result, run_once, reference_fn, op_profile

    # ── Elementwise: auto-resolved, works for ANY valid op name ──
    spec = resolve_op(op_name)
    shape, dtype = spec["shape"], spec["dtype"]
    x_cpu = torch.randn(*shape, dtype=dtype)
    x_gpu = x_cpu.to("cuda")
    fn = spec["fn"]
    ai_result = compute_ai_via_decomposition(fn, x_cpu)
    run_once = lambda: fn(x_gpu)
    reference_fn = fn
    op_profile = dict(op=spec["aten_name"], shape=shape, dtype=dtype)
    return ai_result, run_once, reference_fn, op_profile


def get_op_spec(op_name):
    """Used by build_prompt for semantics text. Compute-heavy ops never
    reach Stage 2, so only elementwise ops need a real spec here."""
    if op_name in _COMPUTE_HEAVY_SPECS:
        return _COMPUTE_HEAVY_SPECS[op_name]
    return resolve_op(op_name)


print("Auto-resolve op system loaded. Any op name in torch / "
      "torch.nn.functional works automatically -- no registry entries needed.")
print("Special-cased (multi-tensor, always compute-bound): "
      + str(list(_COMPUTE_HEAVY_SPECS.keys())))

## 3. Enter your Gemini API key

In [ ]:
from getpass import getpass
os.environ["GEMINI_API_KEY"] = getpass("Paste your Gemini API key: ")
print("Key stored in this session only.")

## 4. Gemini client, prompt builder, compiler, verifier, Nsight profiler (generic across all ops)

In [ ]:
!pip install -q -U google-genai

import re, time, textwrap, tempfile, importlib.util, subprocess
from google import genai as genai_new


def call_gemini(prompt, model=None, max_retries=None):
    """
    model/max_retries default to None here (not GEMINI_MODEL_NAME
    directly) and are resolved INSIDE the function body instead.

    Python evaluates default parameter values ONCE, at function
    definition time -- not at call time. If this used
    model=GEMINI_MODEL_NAME as the default, editing the config cell
    and re-running it LATER would NOT change what this function
    actually uses, since the old value was already baked in when
    this cell first ran. Verified this caused a real bug: switching
    GEMINI_MODEL_NAME to a fresh model after a quota error still
    silently called the OLD, exhausted model.

    Reading GEMINI_MODEL_NAME inside the function body means it
    always reflects whatever the config cell most recently set,
    regardless of cell execution order.
    """
    if model is None:
        model = GEMINI_MODEL_NAME
    if max_retries is None:
        max_retries = MAX_GEMINI_RETRIES

    client = genai_new.Client(api_key=os.environ["GEMINI_API_KEY"])
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(model=model, contents=prompt)
            return response.text
        except Exception as e:
            if "503" in str(e) or "UNAVAILABLE" in str(e):
                wait = 2 ** attempt
                print(f"    Model overloaded (attempt {attempt+1}/{max_retries}), retrying in {wait}s ...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"Gemini still unavailable after {max_retries} retries")


def build_prompt(op_name, spec, op_profile, prior_attempts):
    """Generic across ANY op -- semantics come from resolve_op()'s
    auto-pulled docstring, not a hardcoded per-op description."""
    shape = op_profile["shape"]
    ai, t_ms = op_profile["theoretical_ai"], op_profile["cuda_time_ms"]

    prompt = textwrap.dedent(f"""
    You are an expert GPU kernel engineer.
    Write a high-performance Triton kernel that is mathematically
    equivalent to the PyTorch operator below.

    OPERATOR
    ========
    op              : {op_profile['op']}
    input_shape     : {shape}
    dtype           : float32
    theoretical_AI  : {ai:.4f} FLOPs/byte  (memory-bound, threshold = {AI_THRESHOLD})
    default_latency : {t_ms:.4f} ms

    OPERATOR SEMANTICS
    ==================
    {spec['semantics']}

    YOUR GOAL
    =========
    Write a fused Triton kernel named exactly `graphsynth_kernel` that:
    1. Loads data from DRAM exactly ONCE into SRAM
    2. Performs ALL sub-operations in SRAM — no intermediate DRAM writes
    3. Writes the result back to DRAM exactly ONCE
    4. Selects BLOCK size = triton.next_power_of_2(relevant_dim)

    MANDATORY
    =========
    - The kernel function MUST be named `graphsynth_kernel`
    - Apply appropriate numerical stability if this op involves exp/log/division
      (e.g. subtract max before exp)
    - Do NOT call tl.store() anywhere except the final output write
    - Include a Python launch_kernel(x) wrapper taking ONE tensor argument
      and returning ONE tensor of the same shape

    OUTPUT FORMAT
    =============
    Return ONLY a Python code block.
    ```python
    import triton
    import triton.language as tl
    import torch

    @triton.jit
    def graphsynth_kernel(...):
        ...

    def launch_kernel(x: torch.Tensor) -> torch.Tensor:
        ...
        return output
    ```
    """)
    if prior_attempts:
        prompt += "\nPRIOR ATTEMPT RESULTS\n" + "="*30 + "\n"
        for i, a in enumerate(prior_attempts):
            prompt += f"\nAttempt {i+1}: {a}\n"
    return prompt


def extract_code(response_text):
    m = re.search(r"```python\s*(.*?)```", response_text, re.DOTALL)
    if m:
        return m.group(1).strip()
    m = re.search(r"```\s*(.*?)```", response_text, re.DOTALL)
    return m.group(1).strip() if m else response_text.strip()


def compile_kernel(kernel_code):
    """Writes to a real temp .py file — @triton.jit needs real source
    on disk via inspect.getsource(); exec() into a dict does not work."""
    tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False, prefix="graphsynth_kernel_")
    tmp.write(kernel_code)
    tmp.flush()
    tmp.close()
    try:
        spec = importlib.util.spec_from_file_location("graphsynth_gen", tmp.name)
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        if not hasattr(module, "launch_kernel"):
            return None, "no launch_kernel() function generated"
        return module.launch_kernel, None
    except Exception as e:
        return None, str(e)


EPSILON = {torch.float32: 1e-5, torch.float16: 1e-3, torch.bfloat16: 1e-2}

def generate_test_inputs(shape, dtype, device="cuda"):
    return [
        ("standard randn",    torch.randn(*shape, dtype=dtype, device=device)),
        ("large values x100", torch.randn(*shape, dtype=dtype, device=device) * 100),
        ("small values",      torch.randn(*shape, dtype=dtype, device=device) * 1e-3),
        ("all negative",     -torch.abs(torch.randn(*shape, dtype=dtype, device=device))),
        ("zeros",              torch.zeros(*shape, dtype=dtype, device=device)),
    ]

def verify(kernel_fn, reference_fn, shape, dtype):
    """Manual tolerance check, equivalent to torch.allclose() but also
    reports the actual worst-case error magnitude and nan/inf status."""
    eps = EPSILON.get(dtype, 1e-5)
    test_inputs = generate_test_inputs(shape, dtype)
    all_pass, worst_err = True, 0.0
    for name, x in test_inputs:
        try:
            ref, out = reference_fn(x), kernel_fn(x)
        except Exception as e:
            print(f"    [ERROR] {name:<20} kernel raised: {e}")
            all_pass = False
            continue
        has_nan, has_inf = torch.isnan(out).any().item(), torch.isinf(out).any().item()
        max_err = (ref - out).abs().max().item()
        worst_err = max(worst_err, max_err)
        passed = (max_err < eps) and not has_nan and not has_inf
        print(f"    [{'PASS' if passed else 'FAIL'}] {name:<20} max_err={max_err:.2e}  nan={has_nan}  inf={has_inf}")
        if not passed:
            all_pass = False
    return all_pass, worst_err


NSIGHT_METRICS = [
    "dram__bytes_read.sum", "dram__bytes_write.sum",
    "sm__flops_sp_fma_x2.sum", "gpu__time_duration.sum",
    "lts__t_sector_hit_rate.pct",
]

def write_benchmark_script(kernel_code, shape, dtype_str="float32", n_warmup=10, n_measure=50):
    """header/footer dedented SEPARATELY from kernel_code, then joined —
    dedenting one f-string with kernel_code embedded breaks (kernel_code
    starts at column 0, making the dedent minimum 0, leaving template
    lines indented -> IndentationError). Verified and fixed."""
    header = textwrap.dedent(f"""\
    import torch
    import triton
    import triton.language as tl

    """)
    footer = textwrap.dedent(f"""

    shape = {shape}
    x = torch.randn(*shape, dtype=torch.{dtype_str}, device='cuda')

    for _ in range({n_warmup}):
        out = launch_kernel(x)
    torch.cuda.synchronize()

    for _ in range({n_measure}):
        out = launch_kernel(x)
    torch.cuda.synchronize()
    """)
    script = header + kernel_code + footer
    tmp = tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False, prefix="graphsynth_bench_")
    tmp.write(script)
    tmp.flush()
    return tmp.name

def run_nsight(kernel_code, shape, kernel_name="graphsynth_kernel", dtype_str="float32"):
    script_path = write_benchmark_script(kernel_code, shape, dtype_str)
    cmd = ["ncu", "--metrics", ",".join(NSIGHT_METRICS), "--kernel-name", kernel_name,
           "python", script_path]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
    return result.stdout, result.stderr

def parse_ncu_text(raw_output):
    """Averages across ALL profiled invocations, not just the first —
    --kernel-name filters to one kernel, but the loop calls it n_measure
    times, and ncu reports each call as a separate metric block."""
    metrics = {}
    for metric_name in NSIGHT_METRICS:
        pattern = rf"{re.escape(metric_name)}\s+\S+\s+([\d.,]+)"
        matches = re.findall(pattern, raw_output)
        if matches:
            values = [float(m.replace(",", "")) for m in matches]
            metrics[metric_name] = sum(values) / len(values)
    return metrics

def compute_roofline(raw_metrics, flops_total, peak_bw_gbs):
    """UNIT NOTE: Nsight's text output already scales dram bytes into
    MEGABYTES and time into MICROSECONDS — verified against real output.
    Treating them as raw bytes/nanoseconds silently gave a true_AI of
    ~576000 and a 0.0001ms latency for a kernel that actually ran at
    AI=0.58, latency=0.14ms. Converted explicitly below."""
    dram_read_mb = raw_metrics.get("dram__bytes_read.sum", 0)
    dram_write_mb = raw_metrics.get("dram__bytes_write.sum", 0)
    dram_bytes = (dram_read_mb + dram_write_mb) * 1e6

    time_us = raw_metrics.get("gpu__time_duration.sum", 1)
    time_ns = time_us * 1e3

    achieved_bw = (dram_bytes / (time_ns * 1e-9)) / 1e9 if time_ns else 0
    true_ai = flops_total / max(dram_bytes, 1)
    roofline_pct = (achieved_bw / peak_bw_gbs) * 100
    return dict(true_ai=true_ai, achieved_bw_gbs=achieved_bw, roofline_pct=roofline_pct,
                dram_bytes_mb=dram_bytes/1e6, latency_ms=time_ns/1e6)

print("All Stage 2/3/4 functions loaded (generic across all registered ops).")

## 5. Main loop — run Stage 1-4 for every op in `OPS_TO_TEST`

In [ ]:
# -- Stage 1 for ALL ops FIRST: compute AI + CUDA time + bottleneck_score,
# then RANK before deciding synthesis order -- this is the actual formula
# established in the paper: bottleneck_score = CUDA_time / theoretical_AI
print("=" * 70)
print("STAGE 1 -- profiling and ranking ALL requested ops before synthesis")
print("=" * 70)

stage1_results = {}
for op_name in OPS_TO_TEST:
    try:
        ai_result, run_once, reference_fn, op_profile = build_op_context(op_name)
    except ValueError as e:
        print(f"  {op_name} - {e}")
        continue
    ai, flops, passes, nprims = ai_result
    baseline_us = measure_cuda_time_us(run_once)
    baseline_ms = baseline_us / 1000
    is_compute_bound = ai > AI_THRESHOLD
    score = (baseline_us / ai) if not is_compute_bound else None
    stage1_results[op_name] = dict(
        ai=ai, cuda_us=baseline_us, cuda_ms=baseline_ms, score=score,
        compute_bound=is_compute_bound, ai_result=ai_result,
        run_once=run_once, reference_fn=reference_fn, op_profile=op_profile,
    )

print("\n{:<12}{:>10}{:>12}{:>14}  Classification".format("Op", "AI", "CUDA(ms)", "Score"))
print("-" * 70)
ranked_preview = sorted(
    stage1_results.items(),
    key=lambda kv: (kv[1]["score"] is None, -(kv[1]["score"] or 0))
)
for op_name, r in ranked_preview:
    score_str = "{:.1f}".format(r["score"]) if r["score"] is not None else "--"
    cls = "COMPUTE-BOUND (skip)" if r["compute_bound"] else "MEMORY-BOUND (target)"
    print("{:<12}{:>10.4f}{:>12.4f}{:>14}  {}".format(op_name, r["ai"], r["cuda_ms"], score_str, cls))

synthesis_order = [op for op, r in ranked_preview if not r["compute_bound"]]
skipped_ops = [op for op, r in ranked_preview if r["compute_bound"]]

print("\nSynthesis order (highest bottleneck_score first): {}".format(synthesis_order))
print("Skipped (compute-bound): {}".format(skipped_ops))

results_summary = []

for op_name in skipped_ops:
    r = stage1_results[op_name]
    print("\n{}: COMPUTE-BOUND (AI={:.2f} > {}) -> SKIPPED (cuBLAS/cuDNN already optimal)".format(
        op_name, r["ai"], AI_THRESHOLD))
    results_summary.append(dict(op=op_name, status="SKIPPED (compute-bound)",
                                  ai=r["ai"], score=None,
                                  iterations=0, roofline=None, speedup=None))

for op_name in synthesis_order:
    print("\n" + "=" * 70)
    print("OP: {}  (bottleneck_score={:.1f})".format(op_name, stage1_results[op_name]["score"]))
    print("=" * 70)

    spec = get_op_spec(op_name)
    r = stage1_results[op_name]
    ai, flops, passes, nprims = r["ai_result"]
    baseline_ms = r["cuda_ms"]
    run_once, reference_fn, op_profile = r["run_once"], r["reference_fn"], r["op_profile"]

    print("  Stage 1: AI={:.4f} FLOPs/byte  |  baseline={:.4f} ms  |  score={:.1f}".format(
        ai, baseline_ms, r["score"]))
    print("  MEMORY-BOUND -> dispatching to Stage 2 synthesis")

    op_profile["theoretical_ai"] = ai
    op_profile["cuda_time_ms"] = baseline_ms

    prior_attempts = []
    accepted = False
    final_roofline, final_speedup, iterations_used = None, None, 0

    for iteration in range(MAX_ITERATIONS):
        iterations_used = iteration + 1
        print("\n  -- Iteration {}/{} --".format(iterations_used, MAX_ITERATIONS))

        prompt = build_prompt(op_name, spec, op_profile, prior_attempts)
        try:
            response_text = call_gemini(prompt)
        except Exception as e:
            print("    Gemini error: {}".format(e))
            prior_attempts.append({"error": str(e)})
            continue

        kernel_code = extract_code(response_text)
        kernel_fn, compile_error = compile_kernel(kernel_code)

        if kernel_fn is None:
            print("    Compile failed: {}".format(compile_error))
            prior_attempts.append({"diagnosis": "compile failed: {}".format(compile_error)})
            continue

        verified, worst_err = verify(kernel_fn, reference_fn, spec["shape"], spec["dtype"])
        if not verified:
            print("    Verification FAILED (worst_err={:.2e})".format(worst_err))
            prior_attempts.append({"diagnosis": "verification failed, max_err={:.2e}".format(worst_err)})
            continue
        print("    Verification PASSED (worst_err={:.2e})".format(worst_err))

        stdout, stderr = run_nsight(kernel_code, spec["shape"])
        raw_metrics = parse_ncu_text(stdout)
        if not raw_metrics:
            print("    Nsight profiling failed to produce metrics")
            prior_attempts.append({"diagnosis": "nsight profiling failed"})
            continue

        result = compute_roofline(raw_metrics, flops, PEAK_BW_GBS)
        speedup = baseline_ms / result["latency_ms"] if result["latency_ms"] else 0
        print("    roofline={:.1f}%  speedup={:.2f}x  true_AI={:.4f}".format(
            result["roofline_pct"], speedup, result["true_ai"]))

        if result["roofline_pct"] >= ACCEPT_ROOFLINE_PCT:
            accepted = True
            final_roofline = result["roofline_pct"]
            final_speedup = speedup

            # Save the actual generated Triton kernel to a real,
            # permanent, clearly-named file -- this is the actual
            # deliverable of GraphSynth, not just a console printout
            os.makedirs("generated_kernels", exist_ok=True)
            kernel_path = "generated_kernels/{}_kernel.py".format(op_name)
            with open(kernel_path, "w") as f:
                f.write(kernel_code)

            print("    -> ACCEPTED")
            print("    Kernel saved to: {}".format(kernel_path))
            break
        else:
            prior_attempts.append(result)
            print("    -> below {}% threshold, iterating".format(ACCEPT_ROOFLINE_PCT))

    status = "ACCEPTED" if accepted else "NOT ACCEPTED after {} iter".format(iterations_used)
    results_summary.append(dict(op=op_name, status=status, ai=ai, score=r["score"],
                                  iterations=iterations_used,
                                  roofline=final_roofline, speedup=final_speedup))

## 6. Final results summary

In [ ]:
print("\n\n" + "=" * 98)
print("FINAL RESULTS SUMMARY  (ranked by bottleneck_score = CUDA_time / theoretical_AI)")
print("=" * 98)
print("{:<12}{:<26}{:>8}{:>10}{:>12}{:>12}{:>10}".format(
    "Op", "Status", "AI", "Score", "Iterations", "Roofline%", "Speedup"))
print("-" * 98)
for r in results_summary:
    ai_str = "{:.3f}".format(r["ai"]) if r["ai"] is not None else "--"
    score_str = "{:.1f}".format(r["score"]) if r.get("score") is not None else "--"
    roofline_str = "{:.1f}%".format(r["roofline"]) if r["roofline"] is not None else "--"
    speedup_str = "{:.2f}x".format(r["speedup"]) if r["speedup"] is not None else "--"
    print("{:<12}{:<26}{:>8}{:>10}{:>12}{:>12}{:>10}".format(
        r["op"], r["status"], ai_str, score_str, r["iterations"], roofline_str, speedup_str))
print("=" * 98)

## 7. Show all generated Triton kernels (the actual deliverable)

In [ ]:
import glob

kernel_files = sorted(glob.glob("generated_kernels/*.py"))

if not kernel_files:
    print("No kernels were saved -- either no op was ACCEPTED, or this cell ran before the main loop.")
else:
    print(f"{len(kernel_files)} Triton kernel(s) saved to generated_kernels/\n")
    for path in kernel_files:
        print("=" * 70)
        print(path)
        print("=" * 70)
        with open(path) as f:
            print(f.read())
        print()

print("\nDownload these files: click the folder icon on the left sidebar,")
print("navigate to generated_kernels/, right-click each file -> Download.")

## What to paste back
Copy everything printed from the main loop cell and the final summary
table, and send it back — I'll check the results across all operators.